# 検証つき蒸留の試作（life #45・docs/verified-distill-plan.md §5 の 2〜4・Colab L4 用）

1. 学習用 1,000 問（種 1103）と開発用 300 問を作り、封との 13 文字の並びと意味の近さ（ruri-v3・0.90 以上は捨てる）を確かめる
2. 教師（Qwen3.8-27B UD-Q5_K_M・b11836・思考 OFF・温度 0.7・上限 512・1 問 2 回・並列 16）に答えさせ、`verify.py` で照合する。
   補充は §3 のとおり。問ごとに残った割合が全体で 60% 未満か、どれか 1 種類で 50% 未満なら止める（止める条件 3）
3. 理由の抜き取り 210 件を書き出す（確認役 Codex が手元で点検。4 件以上誤りなら止める・止める条件 4）
4. 学習率 5e-5・1e-4・2e-4 の 3 本を、それぞれ学習前の E4B から学習 → 結合 → Q4_K_M。対照（学習前）も Q4_K_M
5. 4 つを守りの問 40 問と開発用「選ぶ用 150」で測り、§5 の 3 の決まりで選ぶ（満たす物が無ければ止める・止める条件 5）
6. 選んだ物と対照を「確かめる用 150」の境目の問で測る。差が 10 ポイント未満なら止める（止める条件 6）。封 C 24 問は記録だけ

計算の上限は 40 CU（11/02 本人）。各段の結果は `*_JSON` の行と `/content/pilot/log.jsonl` に出す。ランタイムは **L4**。

In [ ]:
# 準備: リポジトリ・GPU・llama.cpp（配布版の実行ファイルと、変換用のソース）
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
!git clone -q --depth 1 -b research/verified-distill https://github.com/Maaa2005/local-rag-agent.git /content/repo
!git -C /content/repo log -1 --format='repo HEAD %H %s'
!git clone -q --depth 1 -b b11836 https://github.com/ggml-org/llama.cpp.git /content/llama.cpp
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /content | tail -1

In [ ]:
%%capture
# Unsloth の公式ノート（unslothai/notebooks 2362e6c・Gemma4_(E4B)-Text）の入れ方そのまま。numpy は読み込み済みの版に戻す
import numpy as _np
_NP_LOADED = _np.__version__
import os, re
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
import importlib.metadata as _md
try: _tv = tuple(int(x) for x in _md.version("torch").split("+")[0].split(".")[:2])
except Exception: _tv = (2, 11)
_torchcodec = {(2, 11): "torchcodec>=0.11.0", (2, 10): "torchcodec==0.10.*", (2, 9): "torchcodec==0.9.*", (2, 8): "torchcodec==0.7.*", (2, 7): "torchcodec==0.5.*"}.get(_tv, "torchcodec>=0.12.0" if _tv >= (2, 12) else "torchcodec")
!pip install "{_torchcodec}"
!pip install --no-deps --upgrade timm
!pip install -q "numpy=={_NP_LOADED}"
# 変換（convert_hf_to_gguf.py）と意味の近さ（ruri-v3）は、それぞれ別の環境で動かす（gguf 0.19 は numpy 2.2.6 以上を求めるため）
!pip install -q uv
!uv venv -q /content/conv-env
!uv pip install -q --python /content/conv-env/bin/python torch --index-url https://download.pytorch.org/whl/cpu
!uv pip install -q --python /content/conv-env/bin/python /content/llama.cpp/gguf-py "transformers==5.5.0" safetensors sentencepiece protobuf
!uv venv -q /content/near-env
!uv pip install -q --python /content/near-env/bin/python torch --index-url https://download.pytorch.org/whl/cpu
!uv pip install -q --python /content/near-env/bin/python sentence-transformers sentencepiece protobuf

In [ ]:
import json, os, pathlib, re, signal, subprocess, sys, time, urllib.request, hashlib, statistics, random
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
sys.path.insert(0, "/content/repo")
from evaluation.distill.gen import make, leak
from evaluation.distill import verify, grade, guard as guardmod
from evaluation.distill.sealed_c import build as sealed_c
from evaluation.teacher_check.score import grade as grade_c

OUTP = pathlib.Path("/content/pilot"); OUTP.mkdir(exist_ok=True)
STAGES = []
T0 = time.time()
def log(kind, obj):
    line = kind + " " + json.dumps(obj, ensure_ascii=False)
    print(line)
    with open(OUTP / "log.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps({"kind": kind, **obj}, ensure_ascii=False) + "\n")
def stage(name, result, **nums):
    rec = {"stage": name, "result": result, "minutes": round((time.time() - T0) / 60, 1), **nums}
    STAGES.append(rec)
    log("STAGE_JSON", rec)
class Stop(Exception):
    pass

GPU = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
if "L4" not in GPU:
    raise RuntimeError(f"L4 でない: {GPU}")
LLAMA_TAG = "b11836"
pre = pathlib.Path("/content/llama-pre"); pre.mkdir(exist_ok=True)
base = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}"
for a in (f"llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz", f"cudart-llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"):
    r = subprocess.run(["bash", "-c", f"set -o pipefail; curl -sfL {base}/{a} | tar xz -C {pre}"], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{a} が取れない: {r.stderr[-300:]}")
SERVER = str(sorted(pre.rglob("llama-server"))[0])
QUANTIZE = str(sorted(pre.rglob("llama-quantize"))[0])
LLAMA_ENV = dict(os.environ, LD_LIBRARY_PATH=":".join(sorted({str(p.parent) for p in pre.rglob("*.so*")}) + [os.environ.get("LD_LIBRARY_PATH", "")]))
REPO_SHA = subprocess.run(["git", "-C", "/content/repo", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

def launch(argv, logf):
    f = open(logf, "w")
    return subprocess.Popen(argv, stdout=f, stderr=subprocess.STDOUT, start_new_session=True, env=LLAMA_ENV)

def kill(p):
    if p is None:
        return
    try:
        os.killpg(p.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    p.wait()

def wait_up(port, p, limit=900):
    t = time.time()
    while time.time() - t < limit and p.poll() is None:
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=3); return True
        except Exception:
            time.sleep(3)
    return False

def chat(port, messages, temperature, max_tokens, stream=False):
    body = {"model": "m", "messages": messages, "temperature": temperature, "max_tokens": max_tokens, "stream": stream}
    req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/chat/completions", json.dumps(body).encode(),
                                 {"Content-Type": "application/json"})
    t = time.time()
    try:
        if not stream:
            res = json.loads(urllib.request.urlopen(req, timeout=900).read())
            ch = res["choices"][0]
            return {"content": ch["message"].get("content") or "", "reasoning": len(ch["message"].get("reasoning_content") or ""),
                    "finish": ch.get("finish_reason"), "usage": res.get("usage"), "seconds": time.time() - t}
        first, text, finish = None, "", None
        with urllib.request.urlopen(req, timeout=900) as r:
            for line in r:
                line = line.decode().strip()
                if not line.startswith("data:") or line == "data: [DONE]":
                    continue
                d = json.loads(line[5:])
                if d.get("choices"):
                    delta = d["choices"][0].get("delta", {}).get("content") or ""
                    if delta and first is None:
                        first = time.time() - t
                    text += delta
                    finish = d["choices"][0].get("finish_reason") or finish
        return {"content": text, "first": first, "seconds": time.time() - t, "finish": finish}
    except Exception as e:   # 失敗は答え無し（不正解）として数える
        return {"content": "", "error": repr(e)[:200], "seconds": time.time() - t, "first": None, "finish": None, "usage": None, "reasoning": 0}

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 24), b""):
            h.update(b)
    return h.hexdigest()

# 問を作る（学習用 1,000・開発用 300 を選ぶ用と確かめる用に）
TRAIN_SEED = 1103
ROWS = make.generate("train", 1000, TRAIN_SEED)
DEV = grade.dev_rows()
DEV_SELECT, DEV_CONFIRM = grade.split_dev(DEV)
GUARD = [json.loads(l) for l in open("/content/repo/evaluation/distill/guard/guard.jsonl", encoding="utf-8")]
SEALED_C = [json.loads(l) for l in open("/content/repo/evaluation/distill/sealed_c/sealed/questions.jsonl", encoding="utf-8")]
sealed13 = leak.sealed_grams() - leak.allowed_grams()
over13 = [r["id"] for r in ROWS + DEV if leak.overlap(r["messages"][1]["content"], sealed13)]
with open(OUTP / "rows.jsonl", "w", encoding="utf-8") as f:
    for r in ROWS + DEV:
        f.write(json.dumps({"id": r["id"], "question": r["question"]}, ensure_ascii=False) + "\n")
log("ROWS_JSON", {"repo": REPO_SHA, "gpu": GPU, "train": len(ROWS), "dev": len(DEV), "select": len(DEV_SELECT),
                  "confirm": len(DEV_CONFIRM), "guard": len(GUARD), "sealed_c": len(SEALED_C), "over13": over13,
                  "train_mix": dict(Counter(r["category"] for r in ROWS))})
stage("準備", "pass" if not over13 else "prep_failure", repo=REPO_SHA)

In [ ]:
# 意味の近さ（ruri-v3・CPU・別の環境 near-env）。封の問と 0.90 以上なら捨てる
r = subprocess.run(["/content/near-env/bin/python", "-m", "evaluation.distill.near", "--rows", str(OUTP / "rows.jsonl"),
                    "--out", str(OUTP / "near.json")], cwd="/content/repo", capture_output=True, text=True,
                   env=dict(os.environ, HF_HUB_DISABLE_PROGRESS_BARS="1", TOKENIZERS_PARALLELISM="false"))
if r.returncode != 0:
    print(r.stdout[-1500:], r.stderr[-3000:]); raise RuntimeError("意味の近さを測れない")
NEAR = json.load(open(OUTP / "near.json", encoding="utf-8"))
drop = set(NEAR["drop"])
ROWS = [x for x in ROWS if x["id"] not in drop]
DEV_SELECT = [x for x in DEV_SELECT if x["id"] not in drop]
DEV_CONFIRM = [x for x in DEV_CONFIRM if x["id"] not in drop]
log("NEAR_JSON", {"summary": NEAR["summary"], "drop": sorted(drop), "train_left": len(ROWS),
                  "select_left": len(DEV_SELECT), "confirm_left": len(DEV_CONFIRM)})
stage("意味の近さ", "pass", dropped=len(drop))

In [ ]:
# 教師の重み
from huggingface_hub import hf_hub_download
TEACHER = hf_hub_download("unsloth/Qwen3.8-27B-GGUF", "Qwen3.8-27B-UD-Q5_K_M.gguf",
                          revision="4ca720788d1e01f1bff70c033e0d0028fd02e502", local_dir="/content/gguf")
stage("教師の重み", "pass", gib=round(os.path.getsize(TEACHER) / 2**30, 2))

In [ ]:
# 教師: 並列 16・1 枠 1,536（入力は最大 465・出力の上限 512 で収まる）。1 問 2 回。照合・重複除き・補充
NP, SLOT = 16, 1536
tp = launch([SERVER, "-m", TEACHER, "-ngl", "999", "-c", str(NP * SLOT), "-np", str(NP), "--jinja", "--reasoning", "off",
             "--host", "127.0.0.1", "--port", "8001"], OUTP / "teacher.log")
if not wait_up(8001, tp):
    print("\n".join(open(OUTP / "teacher.log").read().splitlines()[-30:])); raise RuntimeError("教師が起動しない")

def ask(job):
    r, k = job
    out = chat(8001, r["messages"], 0.7, 512)
    ok, why = verify.accept(r, out["content"], out["finish"], (out["usage"] or {}).get("completion_tokens"))
    return {"id": r["id"], "k": k, "category": r["category"], "ok": ok, "why": why, "answer": out["content"],
            "finish": out["finish"], "tokens": (out["usage"] or {}).get("completion_tokens"),
            "prompt_tokens": (out["usage"] or {}).get("prompt_tokens"), "reasoning": out["reasoning"], "error": out.get("error")}

def answer_all(rows):
    with ThreadPoolExecutor(NP) as ex:
        return list(ex.map(ask, [(r, k) for r in rows for k in range(2)]))

t = time.time()
TEACH = answer_all(ROWS)
teach_sec = time.time() - t

def kept_of(teach):
    by_q = defaultdict(list)
    for x in teach:
        if x["ok"]:
            by_q[x["id"]].append(x["answer"])
    return {q: verify.dedupe(a) for q, a in by_q.items()}

KEPT = kept_of(TEACH)
ROW = {r["id"]: r for r in ROWS}
orig = Counter(r["category"] for r in ROWS)
# 補充（§3）: 残った問の中の割合が目標の 8 割を下回った種類だけ、元の数の 2 倍まで足す（元の数ずつ 2 回まで）
REFILL = {}
for rnd in range(2):
    kept_cat = Counter(ROW[q]["category"] for q in KEPT)
    total = sum(kept_cat.values())
    short = [c for c in orig if kept_cat[c] / total < 0.8 * make.MIX[c]]
    if not short:
        break
    seen = {r["messages"][1]["content"] for r in ROW.values()}
    pool = make.pool("train", TRAIN_SEED + 100 + rnd)
    rng = random.Random(TRAIN_SEED + 200 + rnd)
    extra = []
    for c in short:
        got, tries = 0, 0
        while got < orig[c] and tries < orig[c] * 20:
            tries += 1
            x = make.assemble(rng.choice(pool[c]), "train", rng)
            if x["messages"][1]["content"] in seen or leak.overlap(x["messages"][1]["content"], sealed13):
                continue
            seen.add(x["messages"][1]["content"])
            x["id"] = f"R{rnd}-{c}-{got:04d}"
            extra.append(x); got += 1
        REFILL[f"{rnd}:{c}"] = got
    more = answer_all(extra)
    TEACH += more
    for x in extra:
        ROW[x["id"]] = x
    KEPT = kept_of(TEACH)
teach_sec = time.time() - t
kill(tp)
(OUTP / "teacher.jsonl").write_text("".join(json.dumps(x, ensure_ascii=False) + "\n" for x in TEACH), encoding="utf-8")
asked_q = Counter(ROW[q]["category"] for q in {x["id"] for x in TEACH})
kept_q = Counter(ROW[q]["category"] for q in KEPT)
q_rate = {c: round(kept_q[c] / asked_q[c], 3) for c in asked_q}
overall = len(KEPT) / len({x["id"] for x in TEACH})
TEACH_SUMMARY = {"answers": len(TEACH), "seconds": round(teach_sec), "sec_per_question": round(teach_sec / len({x["id"] for x in TEACH}), 2),
                 "answers_ok": sum(x["ok"] for x in TEACH), "questions_asked": len({x["id"] for x in TEACH}),
                 "questions_ok": len(KEPT), "examples": sum(len(v) for v in KEPT.values()), "q_rate_overall": round(overall, 3),
                 "q_rate": q_rate, "kept_mix": {c: round(kept_q[c] / len(KEPT), 3) for c in kept_q}, "refill": REFILL,
                 "why": dict(Counter(x["why"] for x in TEACH)), "errors": sum(bool(x["error"]) for x in TEACH),
                 "tokens_median": statistics.median([x["tokens"] or 0 for x in TEACH]),
                 "prompt_tokens_max": max(x["prompt_tokens"] or 0 for x in TEACH), "reasoning_any": any(x["reasoning"] for x in TEACH)}
log("TEACHER_JSON", TEACH_SUMMARY)
if overall < 0.60 or min(q_rate.values()) < 0.50:
    stage("教師の答え", "stop_3", q_rate_overall=round(overall, 3))
    raise Stop("止める条件 3: 問ごとに残った割合が足りない")
stage("教師の答え", "pass", seconds=round(teach_sec), questions_ok=len(KEPT), examples=TEACH_SUMMARY["examples"])
os.remove(TEACHER)   # 場所を空ける

In [ ]:
# 理由の抜き取り 210 件（種類ごとに 30 件・種 210）。確認役（Codex）が手元で点検する
rng = random.Random(210)
by_cat = defaultdict(list)
for q, ans in sorted(KEPT.items()):
    for i, a in enumerate(ans):
        by_cat[ROW[q]["category"]].append((q, i, a))
SAMPLE = []
for c in sorted(by_cat):
    items = by_cat[c]
    for q, i, a in rng.sample(items, min(30, len(items))):
        r = ROW[q]
        SAMPLE.append({"sid": f"S{len(SAMPLE) + 1:03d}", "id": q, "category": c, "input": r["messages"][1]["content"],
                       "expected": r["expected"] if c not in ("refusal", "describe") else None,
                       "keywords": r.get("keywords"), "answer": a})
(OUTP / "sample210.jsonl").write_text("".join(json.dumps(x, ensure_ascii=False) + "\n" for x in SAMPLE), encoding="utf-8")
log("SAMPLE_JSON", {"n": len(SAMPLE), "by_category": dict(Counter(x["category"] for x in SAMPLE)),
                    "sha256": sha256(OUTP / "sample210.jsonl"), "bytes": os.path.getsize(OUTP / "sample210.jsonl")})

In [ ]:
# （本人の許可がある時だけ流す）抜き取りを PC に落とす
from google.colab import files
files.download(str(OUTP / "sample210.jsonl"))

In [ ]:
# 学習・結合・変換（学習率ごとに、学習前の E4B から。値は計画 §4）
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import torch, gc
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from huggingface_hub import snapshot_download

EXAMPLES = [{"messages": ROW[q]["messages"] + [{"role": "assistant", "content": a}]} for q, ans in sorted(KEPT.items()) for a in ans[:2]]
BASE_DIR = snapshot_download("unsloth/gemma-4-E4B-it", local_dir="/content/base16")
BATCH = None   # (1 回の件数, 積み重ね)。最初の 1 本の前に 16×1 が入るかを見て決め、3 本で同じにする

def load():
    model, tok = FastModel.from_pretrained(model_name="unsloth/gemma-4-E4B-it", dtype=None, max_seq_length=3072,
                                           load_in_4bit=True, full_finetuning=False)
    model = FastModel.get_peft_model(model, finetune_vision_layers=False, finetune_language_layers=True,
                                     finetune_attention_modules=True, finetune_mlp_modules=True,
                                     r=32, lora_alpha=64, lora_dropout=0.05, bias="none", random_state=3407)
    return model, get_chat_template(tok, chat_template="gemma-4")

def make_trainer(model, tok, rows, lr, bs, ga, max_steps=-1):
    ds = Dataset.from_list(rows)
    ds = ds.map(lambda b: {"text": [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in b["messages"]]}, batched=True)
    tr = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds, eval_dataset=None,
                    args=SFTConfig(dataset_text_field="text", per_device_train_batch_size=bs, gradient_accumulation_steps=ga,
                                   warmup_steps=5, num_train_epochs=2, max_steps=max_steps, learning_rate=lr, logging_steps=5,
                                   optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear", seed=3407,
                                   max_length=3072, report_to="none", disable_tqdm=True))
    return train_on_responses_only(tr), ds

def to_q4(src, name):
    t = time.time()
    r1 = subprocess.run(["/content/conv-env/bin/python", "/content/llama.cpp/convert_hf_to_gguf.py", src, "--outtype", "bf16",
                         "--outfile", f"/content/{name}-bf16.gguf"], capture_output=True, text=True)
    if r1.returncode != 0:
        print(r1.stdout[-1500:], r1.stderr[-3000:]); raise RuntimeError(f"{name} の変換に失敗")
    r2 = subprocess.run([QUANTIZE, f"/content/{name}-bf16.gguf", str(OUTP / f"{name}-q4km.gguf"), "Q4_K_M"],
                        capture_output=True, text=True, env=LLAMA_ENV)
    if r2.returncode != 0:
        print(r2.stdout[-1500:], r2.stderr[-1500:]); raise RuntimeError(f"{name} の量子化に失敗")
    os.remove(f"/content/{name}-bf16.gguf")
    return round(time.time() - t)

from safetensors import safe_open
def first_tensor(d, key_part):
    for f in sorted(pathlib.Path(d).glob("*.safetensors")):
        with safe_open(str(f), "pt") as s:
            for k in s.keys():
                if key_part in k:
                    return k, s.get_tensor(k).float()

# 1 回の件数を決める: 一番長い 16 件で 2 歩だけ 16×1 を試し、入らなければ 8×2
model, tok = load()
longest = sorted(EXAMPLES, key=lambda e: -sum(len(m["content"]) for m in e["messages"]))[:16]
try:
    tr, _ = make_trainer(model, tok, longest, 1e-4, 16, 1, max_steps=2)
    tr.train(); BATCH = (16, 1)
except torch.cuda.OutOfMemoryError:
    BATCH = (8, 2)
peak = round(torch.cuda.max_memory_reserved() / 2**30, 2)
del tr, model, tok; gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
log("BATCH_JSON", {"batch": BATCH, "probe_peak_gib": peak, "examples": len(EXAMPLES)})

TRAINED = {}
def train_one(lr):
    tag = f"lr{lr:g}"
    t = time.time()
    model, tok = load()
    tr, ds = make_trainer(model, tok, EXAMPLES, lr, *BATCH)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    masked_all = all(all(x == -100 for x in ex["labels"]) for ex in tr.train_dataset.select(range(8)))
    bos_id = getattr(tok, "tokenizer", tok).bos_token_id
    bos_count = sum(1 for x in tr.train_dataset[0]["input_ids"] if x == bos_id)
    stats = tr.train()
    losses = [h["loss"] for h in tr.state.log_history if "loss" in h]
    lora_b = sum(float(p.detach().abs().sum()) for n, p in model.named_parameters() if "lora_B" in n)
    train_sec = time.time() - t
    model.save_pretrained(str(OUTP / tag / "adapter")); tok.save_pretrained(str(OUTP / tag / "adapter"))
    t2 = time.time()
    model.save_pretrained_merged("/content/merged16", tok)
    merge_sec = time.time() - t2
    text0 = ds[0]["text"]
    del tr, model, tok, ds; gc.collect(); torch.cuda.empty_cache()
    k1, w1 = first_tensor("/content/merged16", "layers.0.self_attn.q_proj.weight")
    k2, w2 = first_tensor(BASE_DIR, "layers.0.self_attn.q_proj.weight")
    diff = float((w1 - w2).abs().max()) if w1.shape == w2.shape else None
    conv_sec = to_q4("/content/merged16", tag)
    subprocess.run(["rm", "-rf", "/content/merged16"])
    res = {"lr": lr, "tag": tag, "examples": len(EXAMPLES), "batch": BATCH, "trainable": trainable, "masked_all": masked_all,
           "bos_count": bos_count, "loss_first": losses[0] if losses else None, "loss_last": losses[-1] if losses else None,
           "loss_10_or_more": bool(losses and losses[0] >= 10), "lora_b_abs_sum": lora_b, "max_abs_diff": diff,
           "train_runtime": stats.metrics.get("train_runtime"), "train_sec": round(train_sec), "merge_sec": round(merge_sec),
           "conv_sec": conv_sec, "peak_gib": round(torch.cuda.max_memory_reserved() / 2**30, 2),
           "gguf_sha256": sha256(OUTP / f"{tag}-q4km.gguf"), "text0": text0}
    ok = trainable and not masked_all and bos_count == 1 and lora_b > 0 and losses and losses[0] == losses[0] and diff
    TRAINED[lr] = res
    log("TRAIN_JSON", {k: v for k, v in res.items() if k != "text0"})
    stage(f"学習 {tag}", "pass" if ok else "prep_failure", train_sec=round(train_sec))
    torch.cuda.reset_peak_memory_stats()

In [ ]:
train_one(5e-5)

In [ ]:
train_one(1e-4)

In [ ]:
train_one(2e-4)

In [ ]:
# 対照（学習していない E4B）も同じ手順で Q4_K_M
ctrl_sec = to_q4(BASE_DIR, "control")
log("CONTROL_JSON", {"conv_sec": ctrl_sec, "gguf_sha256": sha256(OUTP / "control-q4km.gguf")})
stage("対照の変換", "pass", conv_sec=ctrl_sec)

In [ ]:
# 測る: 守りの問 40（並列 1・流しながら時間を測る）と、開発用「選ぶ用」（並列 4）。温度 0.1・上限 1,024（本番の設定）
ANS = {}
def run_model(tag, sets):
    gguf = str(OUTP / f"{tag}-q4km.gguf")
    out = {}
    sp = launch([SERVER, "-m", gguf, "-ngl", "999", "-c", "4096", "-np", "1", "--jinja", "--reasoning", "off",
                 "--host", "127.0.0.1", "--port", "8002"], OUTP / f"{tag}-guard.log")
    if not wait_up(8002, sp):
        print("\n".join(open(OUTP / f"{tag}-guard.log").read().splitlines()[-30:])); raise RuntimeError(f"{tag} が起動しない")
    if tag != "control":
        text0 = TRAINED[float(tag[2:])]["text0"]
        req = urllib.request.Request("http://127.0.0.1:8002/apply-template", json.dumps({"messages": EXAMPLES[0]["messages"][:-1]}).encode(),
                                     {"Content-Type": "application/json"})
        sp_body = json.loads(urllib.request.urlopen(req).read())["prompt"].removeprefix("<bos>")
        out["template_ok"] = text0.removeprefix("<bos>").startswith(sp_body)
    if "guard" in sets:
        g = {}
        for row in GUARD:
            o = chat(8002, row["messages"], 0.1, 1024, stream=True)
            g[row["id"]] = {"answer": o["content"], "pass": guardmod.judge(row, o["content"]), "first": o["first"], "seconds": o["seconds"]}
        out["guard"] = g
    kill(sp)
    for name in [s for s in sets if s != "guard"]:
        rows = {"select": DEV_SELECT, "confirm": [r for r in DEV_CONFIRM if r["category"] == "boundary"]}.get(name)
        msgs = {r["id"]: r["messages"] for r in rows} if rows else {r["id"]: sealed_c.prompt_messages(r) for r in SEALED_C}
        sp = launch([SERVER, "-m", gguf, "-ngl", "999", "-c", str(4 * 4096), "-np", "4", "--jinja", "--reasoning", "off",
                     "--host", "127.0.0.1", "--port", "8003"], OUTP / f"{tag}-{name}.log")
        if not wait_up(8003, sp):
            raise RuntimeError(f"{tag} が起動しない（{name}）")
        with ThreadPoolExecutor(4) as ex:
            got = dict(zip(msgs, ex.map(lambda m: chat(8003, m, 0.1, 1024)["content"], msgs.values())))
        kill(sp)
        out[name] = got
    ANS.setdefault(tag, {}).update(out)
    (OUTP / f"answers-{tag}.json").write_text(json.dumps(ANS[tag], ensure_ascii=False), encoding="utf-8")

t = time.time()
for tag in ["control"] + [TRAINED[lr]["tag"] for lr in sorted(TRAINED)]:
    run_model(tag, ["guard", "select"])
eval_sec = time.time() - t
ctrl_sec_by = {i: v["seconds"] for i, v in ANS["control"]["guard"].items()}
CANDS = []
for lr in sorted(TRAINED):
    tag = TRAINED[lr]["tag"]
    g = grade.guard_ok(GUARD, {i: v["pass"] for i, v in ANS[tag]["guard"].items()},
                       {i: v["seconds"] for i, v in ANS[tag]["guard"].items()}, ctrl_sec_by)
    acc = grade.accuracy(DEV_SELECT, ANS[tag]["select"])
    CANDS.append({"lr": lr, "tag": tag, "guard_ok": g["ok"], "guard": g, "select_acc": acc, "template_ok": ANS[tag]["template_ok"]})
CHOSEN = grade.choose(CANDS)
log("SELECT_JSON", {"control_select_acc": grade.accuracy(DEV_SELECT, ANS["control"]["select"]),
                    "control_guard_pass": sum(v["pass"] for v in ANS["control"]["guard"].values()),
                    "cands": CANDS, "chosen": CHOSEN and CHOSEN["tag"], "eval_sec": round(eval_sec)})
if CHOSEN is None:
    stage("選ぶ", "stop_5")
    raise Stop("止める条件 5: 3 本とも守りの問の条件を満たさない")
stage("選ぶ", "pass", chosen=CHOSEN["tag"])

In [ ]:
# 続けるかの確認: 確かめる用の境目の問（選んだ物と対照）。封 C 24 問は記録だけ
t = time.time()
for tag in ("control", CHOSEN["tag"]):
    run_model(tag, ["confirm", "sealed_c"])
conf_rows = [r for r in DEV_CONFIRM if r["category"] == "boundary"]
acc_c = grade.accuracy(conf_rows, ANS["control"]["confirm"])
acc_s = grade.accuracy(conf_rows, ANS[CHOSEN["tag"]]["confirm"])
def c_score(tag):
    gs = [grade_c(ANS[tag]["sealed_c"].get(r["id"]), r["expected_conclusion"], r["expected_articles"]) for r in SEALED_C]
    return {"conclusion_ok": sum(g["conclusion_ok"] for g in gs), "ok": sum(g["ok"] for g in gs), "n": len(gs)}
CONFIRM = {"n": len(conf_rows), "control_acc": acc_c, "chosen_acc": acc_s, "diff_pp": round((acc_s - acc_c) * 100, 1),
           "sealed_c": {"control": c_score("control"), "chosen": c_score(CHOSEN["tag"])}, "seconds": round(time.time() - t)}
log("CONFIRM_JSON", CONFIRM)
if acc_s - acc_c < 0.10:
    stage("続けるかの確認", "stop_6", diff_pp=CONFIRM["diff_pp"])
    raise Stop("止める条件 6: 確かめる用の境目の正答率が対照より 10 ポイント以上高くない")
stage("続けるかの確認", "pass", diff_pp=CONFIRM["diff_pp"])

In [ ]:
# まとめ
SUMMARY = {"repo": REPO_SHA, "gpu": GPU, "stages": STAGES, "teacher": TEACH_SUMMARY,
           "trained": {TRAINED[lr]["tag"]: {k: v for k, v in TRAINED[lr].items() if k != "text0"} for lr in TRAINED},
           "chosen": CHOSEN and CHOSEN["tag"], "total_minutes": round((time.time() - T0) / 60, 1)}
log("SUMMARY_JSON", SUMMARY)